In [19]:
import numpy as np
import pandas as pd

In [20]:
df = pd.read_csv('/home/radit/MachineLearning/Star-Correction/data/processed/data_final.csv')

In [21]:
import pandas as pd
from langdetect import detect, LangDetectException

MIN_LENGTH = 20

def is_indonesian(text):
    text = str(text).strip()
    if len(text) < MIN_LENGTH:
        return False  # teks terlalu pendek, langsung buang
    try:
        return detect(text) == 'id'
    except LangDetectException:
        return False

df['valid'] = df['text'].apply(is_indonesian)

valid_df   = df[df['valid']].drop(columns='valid')
removed_df = df[~df['valid']].drop(columns='valid')

print(f"Total data  : {len(df)}")
print(f"Dibuang     : {len(removed_df)}")
print(f"Tersisa     : {len(valid_df)}")

Total data  : 21593
Dibuang     : 6206
Tersisa     : 15387


In [ ]:
# pip install nlpaug
import nlpaug.augmenter.word as naw
import pandas as pd
from sklearn.utils import resample

df = valid_df

# Augmenter: sinonim kata (cocok untuk Bahasa Indonesia)
aug = naw.SynonymAug(aug_src='wordnet')

def augment_text(text, n=1):
    """Generate n variasi teks baru"""
    results = []
    for _ in range(n):
        try:
            augmented = aug.augment(text)
            results.append(augmented[0] if isinstance(augmented, list) else augmented)
        except:
            results.append(text)
    return results

# Hitung berapa yang perlu ditambah
target = df['sentiment_label'].value_counts().max()
df_result = [df]  # mulai dari data asli

for label in ['Negative', 'Neutral']:
    df_label = df[df['sentiment_label'] == label]
    needed   = target - len(df_label)
    
    print(f"Augmenting '{label}': perlu tambah {needed} data...")
    
    # Sample teks yang akan di-augment
    samples = df_label.sample(n=needed, replace=True, random_state=42)
    
    augmented_texts = []
    for text in samples['text']:
        aug_text = augment_text(text, n=1)[0]
        augmented_texts.append(aug_text)
    
    df_aug = samples.copy()
    df_aug['text'] = augmented_texts
    df_result.append(df_aug)

df_balanced = pd.concat(df_result).sample(frac=1, random_state=42).reset_index(drop=True)

print("\nDistribusi akhir:")
print(df_balanced['sentiment_label'].value_counts())
df_balanced.to_csv('../data/preprocessed/data_augmented.csv', index=False)

Augmenting 'Negative': perlu tambah 9812 data...


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/radit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/radit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/radit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/radit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/radit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading packag

Augmenting 'Neutral': perlu tambah 10534 data...


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/radit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/radit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/radit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/radit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/radit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading packag


Distribusi akhir:
sentiment_label
Neutral     11911
Positive    11911
Negative    11911
Name: count, dtype: int64
